# HDMI GUI single-shot display

Single-cell notebook that programs the integrated AudioLab HDMI path and pushes the compact-v2 800x480 GUI onto the 5-inch LCD. Run the cell once; the FPGA keeps scanning out the parked framebuffer until the kernel is restarted or another overlay is loaded.

Compared to `HdmiGui.ipynb`, this notebook has no live animation loop, no `ipywidgets` dropdowns, and no resource monitor: those layers were the most common reason the cell appeared to hang without ever showing the UI. This one renders one frame, writes it to VDMA, prints status, and exits.

## PLL-edge caveat

Phase 6I (`DECISIONS.md` D25) runs HDMI at `40.000 MHz` pixel clock, which puts the `rgb2dvi v1.4` `kClkRange=3` internal MMCM at the absolute lower edge of its valid VCO band (`800 MHz`). After a PYNQ-Z2 power-on the first `Overlay(..., download=True)` locks the PLL cleanly; a second `download=True` re-load in the same session can knock the PLL out and drop the LCD to white / "no signal" even with VDMA and VTC reporting correct state. To avoid that, this cell asks `pynq.PL` whether `audio_lab.bit` is already programmed; if it is, the cell attaches with `AudioLabOverlay(download=False)` and leaves the running rgb2dvi PLL alone. It does NOT do any MMIO read before the overlay is constructed, because reading a PL address while the FPGA is blank can hang the AXI bus and kill the Jupyter kernel (you would see `The kernel appears to have died. It will restart automatically.`).

If the LCD ever goes white, power-cycle the PYNQ-Z2 and run this cell once.

## Pre-flight assertions

- HDMI IPs (`axi_vdma_hdmi`, `v_tc_hdmi`) are present in `overlay.ip_dict`.
- `v_tc_hdmi GEN_ACTSZ` reads `0x02580320` (V=600, H=800 — Phase 6I VESA SVGA, `DECISIONS.md` D25). A mismatch means a stale bit is loaded from one of the five on-board locations; the error message lists all five so the user can sync them.
- No VDMA error bits are asserted.

After this cell finishes, the 800x480 UI is in framebuffer rows `0..479` and rows `480..599` are black. Re-run the cell to redraw with the current `AppState`.

In [ ]:
# HDMI GUI single-shot display (Phase 6I, DECISIONS.md D25).
#
# Smart behaviour:
#   - First run after PYNQ-Z2 power-on: pynq.PL reports no bit loaded
#     (or a different bit), so this cell calls `AudioLabOverlay()` with
#     the default `download=True` and programs the FPGA fresh.
#   - Subsequent re-runs in the same Jupyter session: pynq.PL reports
#     the audio_lab.bit is already loaded, so this cell attaches with
#     `AudioLabOverlay(download=False)`. That avoids re-asserting the
#     FPGA configuration pipeline and keeps the rgb2dvi MMCM PLL lock
#     intact at the 40 MHz / VCO 800 MHz lower edge.
#     Memory: `rgb2dvi-pll-edge-at-40mhz`.
#
# Operating notes:
#   - If the LCD does drop to white between runs, power-cycle the
#     PYNQ-Z2 and run this cell once.
#   - The 800x480 compact-v2 UI lands at framebuffer (0, 0); the
#     bottom 120 rows of the 800x600 framebuffer stay black.
#   - The cell does NOT probe MMIO before the overlay is loaded —
#     reading a PL address while the FPGA is blank can hang the AXI
#     bus and kill the Jupyter kernel.

import os
import sys
PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
for path in (PROJECT_ROOT, PROJECT_ROOT + "/GUI"):
    if path not in sys.path:
        sys.path.insert(0, path)

from pynq import PL, MMIO
from audio_lab_pynq import AudioLabOverlay
from audio_lab_pynq.hdmi_backend import AudioLabHdmiBackend
from pynq_multi_fx_gui import AppState, render_frame_800x480_compact_v2

EXPECTED_VTC_ACTSZ = 0x02580320  # SVGA 800x600: (V << 16) | H

# ---------- 1. decide whether to re-download the bit ---------------
# pynq.PL.bitfile_name is None / "" when no overlay has been loaded
# in this session, otherwise it holds the absolute path to the .bit
# currently programmed. Touching this attribute does NOT do any PL
# bus access, so it is safe even when the FPGA is blank.
try:
    loaded_path = PL.bitfile_name or ""
except Exception:
    loaded_path = ""
loaded_basename = os.path.basename(loaded_path)
already_loaded = (loaded_basename == "audio_lab.bit")

if already_loaded:
    print("pynq.PL reports audio_lab.bit already loaded ({}).".format(loaded_path))
    print("Attaching with download=False to preserve rgb2dvi PLL lock.")
    overlay = AudioLabOverlay(download=False)
else:
    print("pynq.PL bitfile is {!r}; loading Phase 6I C2 SVGA bit fresh.".format(
        loaded_path))
    overlay = AudioLabOverlay()  # default download=True

# ---------- 2. smoke -----------------------------------------------
print("AudioLabOverlay attached")
print("  ADC HPF :", bool(overlay.codec.get_adc_hpf_state()))
print("  R19     :", "0x{:02x}".format(int(overlay.codec.R19_ADC_CONTROL[0]) & 0xFF))
ip_keys = set(getattr(overlay, "ip_dict", {}).keys())
print("  axi_vdma_hdmi in ip_dict:", "axi_vdma_hdmi" in ip_keys)
print("  v_tc_hdmi     in ip_dict:", "v_tc_hdmi" in ip_keys)
assert "axi_vdma_hdmi" in ip_keys and "v_tc_hdmi" in ip_keys, \
    "HDMI IPs missing from the overlay; check the bit is the Phase 6I build"

# ---------- 3. VTC mode sanity (SVGA 800x600), now SAFE to read --
vtc_mm = MMIO(int(overlay.ip_dict["v_tc_hdmi"]["phys_addr"]), 0x1000)
gen_actsz = vtc_mm.read(0x60)
v_active = (gen_actsz >> 16) & 0x1FFF
h_active = gen_actsz & 0x1FFF
print("VTC GEN_ACTSZ (0x60) = 0x{:08x}  -> V_active={} H_active={}".format(
    gen_actsz, v_active, h_active))
if (v_active, h_active) != (600, 800):
    raise RuntimeError(
        "VTC active size is V={} H={}, expected V=600 H=800 (SVGA). "
        "Stale bit copy. Sync all five PYNQ-Z2 locations:\n"
        "  /home/xilinx/Audio-Lab-PYNQ/hw/Pynq-Z2/bitstreams/audio_lab.bit\n"
        "  /home/xilinx/Audio-Lab-PYNQ/audio_lab_pynq/bitstreams/audio_lab.bit\n"
        "  /usr/local/lib/python3.6/dist-packages/audio_lab_pynq/bitstreams/audio_lab.bit\n"
        "  /home/xilinx/jupyter_notebooks/audio_lab/bitstreams/audio_lab.bit\n"
        "  /usr/local/lib/python3.6/dist-packages/pynq/overlays/audio_lab/audio_lab.bit"
        .format(v_active, h_active))

# ---------- 4. render the compact-v2 GUI ---------------------------
state = AppState()
state.preset_id = "P1"
state.preset_name = "DEFAULT"
state.selected_fx = "TUBE SCREAMER"
state.selected_effect = 2
state.pedal_model_label = "TUBE SCREAMER"
state.amp_model_label = "HIGH GAIN STACK"
state.cab_model_label = "2x12 COMBO"
state.in_level = 0.62
state.out_level = 0.58
frame = render_frame_800x480_compact_v2(state, theme="pipboy-green")
print("Rendered compact-v2 frame :", frame.shape, frame.dtype)
assert frame.shape == (480, 800, 3) and str(frame.dtype) == "uint8"

# ---------- 5. push into HDMI framebuffer --------------------------
backend = AudioLabHdmiBackend(overlay)
print("HDMI backend framebuffer  :", backend.width, "x", backend.height,
      "(SVGA 800x600 expected)")
backend.start(frame, placement="manual", offset_x=0, offset_y=0)
status = backend.status()
last = status.get("last_frame_write", {}) or {}
fb = last.get("framebuffer_copied_region", {}) or {}
print("UI placed at framebuffer  : x={x0}..{x1} y={y0}..{y1}".format(
    **{k: fb.get(k, "?") for k in ("x0", "x1", "y0", "y1")}))
print("Bottom black band         : y={}..{}".format(
    fb.get("y1", "?"), backend.height - 1))
print("Placement / offset        : {} / ({}, {})".format(
    last.get("placement"), last.get("offset_x"), last.get("offset_y")))

# ---------- 6. status ----------------------------------------------
errs = backend.errors()
print("VDMA dmacr / dmasr        : {} / {}".format(
    status.get("vdma_dmacr"), status.get("vdma_dmasr")))
print("VDMA hsize / stride / vsize: {} / {} / {}".format(
    status.get("vdma_hsize"), status.get("vdma_stride"), status.get("vdma_vsize")))
print("VDMA error bits           :", errs)
assert not (errs.get("dmainterr") or errs.get("dmaslverr") or errs.get("dmadecerr")), \
    "VDMA error bits asserted; check rgb2dvi clocking and the framebuffer"

print()
if already_loaded:
    print("OK - re-attached to the running C2 overlay; UI refreshed without "
          "touching the rgb2dvi PLL.")
else:
    print("OK - fresh overlay downloaded; LCD should now show the compact-v2 "
          "Pip-Boy UI.")
print("Re-run this cell to redraw with the current AppState.")
